In [6]:
import numpy as np
from scipy.optimize import fsolve, root

import xarray as xr
import pandas as pd

# import predefined functions wich solve the box model
import fct_BoxModel as fbm

import imp
imp.reload(fbm)

<module 'fct_BoxModel' from '/mnt/c/Users/louis/Documents/GitHub/Low-Dimensionnal-Model-2025/fct_BoxModel.py'>

### Importing the input dataset

In [7]:
# inputs from BoxToReal.ipynb
inputs = xr.open_dataset("model_input_dataset.nc")
index_isf = inputs.Nisf.values
name_isf = inputs.name_isf.values

In [8]:
# Define reference properties and parameters 

#EOS
T_star = 0 # °C  (PICO)
S_star = 34 # PSU   (PICO)
rho_star = 1033 # kg/m^3   (PICO)
alpha = 7.5e-5 # /°C   (PICO)
beta = 7.7e-4 # /PSU (PICO)

def EOS(T,S):
    return rho_star*(1-alpha*(T-T_star)+beta*(S-S_star))
    
#Liquidus
la = -0.0572 # °C/PSU   (PICO)
lb = 0.0788 # °C   (PICO)
lc = 7.59e-4 # °C/m   (PICO->Burgard)

def liquidus(S,h):
    return la*S+lb-lc*h

#Water properties
L = 3.34e5 # J/kg   (PICO)
c_star = 3974 # J/kg/°C   (PICO)
lambd = L/c_star # 84 K

#Ice properties
rho_i = 910 # kg/m^3   (PICO)
nu = rho_i/rho_star #0.88

#PICO parameters
# Reese 2018
C = 1e6 #m^6/s/kg
gammaT = 2e-5 #m/s

#Vertical mixing
# ~Olbers & Hellmer 2010
kappa_diff = 1e-7  # m/s
kappa_conv = 1e-3 # m/s

#Polynia
g = 20

# fraction of q_cav that sinks
r = 0.

# For DSW outflow
C_DSW = 4e6 #m^6/s/kg

### Creating dataset containing the outputs of the box model

In [9]:
Dataset= xr.Dataset(
    {
        "name_isf": (["Nisf"],name_isf,{"descr":"Name of the ice shelves"}),
        "Tp": (["Nisf","regime"],np.zeros((9,3)),{"descr":"°C"}),
        "Sp": (["Nisf","regime"],np.zeros((9,3)),{"descr":"PSU"}),
        "Td": (["Nisf","regime"],np.zeros((9,3)),{"descr":"°C"}),
        "Sd": (["Nisf","regime"],np.zeros((9,3)),{"descr":"PSU"}),
        "Tc": (["Nisf","regime","box"],np.zeros((9,3,5)),{"descr":"°C"}),
        "Sc": (["Nisf","regime","box"],np.zeros((9,3,5)),{"descr":"PSU"}),
        "melt": (["Nisf","regime"],np.zeros((9,3)),{"descr":"m/30d"}),
        "overt": (["Nisf","regime"],np.zeros((9,3)),{"descr":"Sv"}),
        "qAABW": (["Nisf"],np.zeros(9),{"descr":"Sv"}),
        
    },
    coords={
        "Nisf": index_isf,
        "box": np.arange(1,5+1),
        "regime": np.array(['diff','conv','conv_AABW'])
    },
    attrs={
        "Global": "Dataset containing the output parameters for the box model",
        "Nisf": "Index of the ice shelves (from Burgard 2022)",
        "box": "box number (but not ntotal box number)",
    }
)

### Filling outputs in the diffusive regime

In [10]:
kappa = kappa_diff

AABW = False

for index in index_isf:
    print(' ')
    print(inputs.name_isf.sel(Nisf=index).values)

    nbox = int(inputs.Nbox.sel(Nisf=index).values)
    Ac = inputs.Ac.sel(Nisf=index).values #m^2

    xi_ref = np.mean(inputs.xi.sel(Nisf=index).values)/30/24/3600 *2/3 #m/s 2/3 factor for yearly average
    Ap_ref = np.mean(inputs.Ap.sel(Nisf=index).values) *2/3 #m 2/3 factor for yearly average
    T_CDW_ref =  np.mean(inputs.T_CDW.sel(Nisf=index).values) #°C
    
    frac = inputs.frac.sel(Nisf=index).values
    depth = inputs.depth.sel(Nisf=index).values
    T_surf_ref = np.mean(inputs.T_surf.sel(Nisf=index).values)
    S_surf_ref = np.mean(inputs.S_surf.sel(Nisf=index).values)
    S_CDW_ref = np.mean(inputs.S_CDW.sel(Nisf=index).values)

    
    guess= np.array([0,0,34.5,34.5])

    args = (nbox,C,gammaT,Ac,xi_ref,T_CDW_ref,S_CDW_ref,Ap_ref,kappa,g,frac,depth,r)

    X, info, ier, msg = fsolve(fbm.BoxModel, guess, args=args , full_output=True,)
    print(msg)

    sigma = fbm.compute_Sigma(X)
    print(sigma<0)


    Tp, Td, Sp, Sd = X

    if sigma<0:
        Tc ,Sc ,m,q = fbm.PICO(Td,Sd, nbox, C, gammaT, Ac, frac, depth)
        m_avg = fbm.compute_m_avg(m,frac,nbox)

        Tc = np.concatenate((Tc,np.zeros(5-len(Tc))))
        Sc = np.concatenate((Sc,np.zeros(5-len(Tc))))

        Dataset['Tp'].loc[index,"diff"] = Tp
        Dataset['Td'].loc[index,"diff"] = Td
        Dataset['Sp'].loc[index,"diff"] = Sp
        Dataset['Sd'].loc[index,"diff"] = Sd
        Dataset['Tc'].loc[index,"diff"] = Tc
        Dataset['melt'].loc[index,"diff"] = m_avg
        Dataset['overt'].loc[index,"diff"] = q/1e6
        
        
        

 
Ross
The solution converged.
True
 
Filchner-Ronne
The solution converged.
True
 
Amery
The solution converged.
True
 
Dotson
The solution converged.
False
 
Pine Island
The solution converged.
True
 
Riiser-Larsen
The solution converged.
True
 
Roi Baudouin
The solution converged.
True
 
Totten
The solution converged.
True
 
Moscow Univ.
The solution converged.
True


/mnt/c/Users/louis/Documents/GitHub/Low-Dimensionnal-Model-2025/fct_BoxModel.py:44: RuntimeWarning: invalid value encountered in sqrt
  x0 = -g1[0]/(2*C*rho_star*(beta*s-alpha))+np.sqrt((g1[0]/(2*C*rho_star*(beta*s-alpha)))**2-g1[0]*T_st_0/(C*rho_star*(beta*s-alpha)))


### Filling outputs in the convective regime

In [11]:
kappa = kappa_conv

AABW = False

for index in index_isf:
    print(' ')
    print(inputs.name_isf.sel(Nisf=index).values)

    nbox = int(inputs.Nbox.sel(Nisf=index).values)
    Ac = inputs.Ac.sel(Nisf=index).values #m^2

    xi_ref = np.mean(inputs.xi.sel(Nisf=index).values)/30/24/3600 *2/3 #m/s 2/3 factor for yearly average
    Ap_ref = np.mean(inputs.Ap.sel(Nisf=index).values) *2/3 #m 2/3 factor for yearly average
    T_CDW_ref =  np.mean(inputs.T_CDW.sel(Nisf=index).values) #°C
    
    frac = inputs.frac.sel(Nisf=index).values
    depth = inputs.depth.sel(Nisf=index).values
    T_surf_ref = np.mean(inputs.T_surf.sel(Nisf=index).values)
    S_surf_ref = np.mean(inputs.S_surf.sel(Nisf=index).values)
    S_CDW_ref = np.mean(inputs.S_CDW.sel(Nisf=index).values)

    
    guess= np.array([0,0,34.5,34.5])

    args = (nbox,C,gammaT,Ac,xi_ref,T_CDW_ref,S_CDW_ref,Ap_ref,kappa,g,frac,depth,r)

    X, info, ier, msg = fsolve(fbm.BoxModel, guess, args=args, full_output=True)
    print(msg)

    sigma = fbm.compute_Sigma(X)
    print(sigma>0)


    Tp, Td, Sp, Sd = X

    if sigma>0:
        Tc ,Sc ,m,q = fbm.PICO(Td,Sd, nbox, C, gammaT, Ac, frac, depth)
        m_avg = fbm.compute_m_avg(m,frac,nbox)

        Tc = np.concatenate((Tc,np.zeros(5-len(Tc))))
        Sc = np.concatenate((Sc,np.zeros(5-len(Tc))))

        Dataset['Tp'].loc[index,"conv"] = Tp
        Dataset['Td'].loc[index,"conv"] = Td
        Dataset['Sp'].loc[index,"conv"] = Sp
        Dataset['Sd'].loc[index,"conv"] = Sd
        Dataset['Tc'].loc[index,"conv"] = Tc
        Dataset['melt'].loc[index,"conv"] = m_avg
        Dataset['overt'].loc[index,"conv"] = q/1e6

 
Ross
The solution converged.
True
 
Filchner-Ronne
The solution converged.
True
 
Amery
The solution converged.
True
 
Dotson
The solution converged.
True
 
Pine Island
The solution converged.
True
 
Riiser-Larsen
The solution converged.
True
 
Roi Baudouin
The solution converged.
True
 
Totten
The solution converged.
True
 
Moscow Univ.
The solution converged.
False


### Filling outputs in the convective regime, with AABW parameterization

In [13]:
kappa = kappa_conv

AABW = True

for index in index_isf:
    print(' ')
    print(inputs.name_isf.sel(Nisf=index).values)

    nbox = int(inputs.Nbox.sel(Nisf=index).values)
    Ac = inputs.Ac.sel(Nisf=index).values #m^2

    xi_ref = np.mean(inputs.xi.sel(Nisf=index).values)/30/24/3600 *2/3 #m/s 2/3 factor for yearly average
    Ap_ref = np.mean(inputs.Ap.sel(Nisf=index).values) *2/3 #m 2/3 factor for yearly average
    T_CDW_ref =  np.mean(inputs.T_CDW.sel(Nisf=index).values) #°C
    
    frac = inputs.frac.sel(Nisf=index).values
    depth = inputs.depth.sel(Nisf=index).values
    T_surf_ref = np.mean(inputs.T_surf.sel(Nisf=index).values)
    S_surf_ref = np.mean(inputs.S_surf.sel(Nisf=index).values)
    S_CDW_ref = np.mean(inputs.S_CDW.sel(Nisf=index).values)

    
    guess= np.array([0,0,34.5,34.5])

    args = (nbox,C,gammaT,Ac,xi_ref,T_CDW_ref,S_CDW_ref,Ap_ref,kappa,g,frac,depth,r, True, C_DSW, T_surf_ref, S_surf_ref)

    X, info, ier, msg = fsolve(fbm.BoxModel, guess, args=args, full_output=True)
    print(msg)

    sigma = fbm.compute_Sigma(X)
    print(sigma>0)


    Tp, Td, Sp, Sd = X

    if sigma>0:
        Tc ,Sc ,m,q = fbm.PICO(Td,Sd, nbox, C, gammaT, Ac, frac, depth)
        m_avg = fbm.compute_m_avg(m,frac,nbox)

        Tc = np.concatenate((Tc,np.zeros(5-len(Tc))))
        Sc = np.concatenate((Sc,np.zeros(5-len(Tc))))

        qAABW = fbm.compute_DSW(Td, Sd, C_DSW, T_CDW_ref, S_CDW_ref)

        Dataset['Tp'].loc[index,"conv_AABW"] = Tp
        Dataset['Td'].loc[index,"conv_AABW"] = Td
        Dataset['Sp'].loc[index,"conv_AABW"] = Sp
        Dataset['Sd'].loc[index,"conv_AABW"] = Sd
        Dataset['Tc'].loc[index,"conv_AABW"] = Tc
        Dataset['melt'].loc[index,"conv_AABW"] = m_avg
        Dataset['overt'].loc[index,"conv_AABW"] = q/1e6
        Dataset['qAABW'].loc[index] = qAABW/1e6

 
Ross
The solution converged.
True
 
Filchner-Ronne
The solution converged.
True
 
Amery
The solution converged.
True
 
Dotson
The solution converged.
True
 
Pine Island
The solution converged.
True
 
Riiser-Larsen
The solution converged.
True
 
Roi Baudouin
The solution converged.
True
 
Totten
The solution converged.
True
 
Moscow Univ.
The solution converged.
False


### Creating Dataset

In [16]:
Dataset.to_netcdf("model_output_dataset.nc") # uncomment to create the output dataset

In [17]:
Dataset

<xarray.Dataset> Size: 4kB
Dimensions:   (Nisf: 9, regime: 3, box: 5)
Coordinates:
  * Nisf      (Nisf) int64 72B 10 11 31 52 66 43 45 48 33
  * box       (box) int64 40B 1 2 3 4 5
  * regime    (regime) <U9 108B 'diff' 'conv' 'conv_AABW'
Data variables:
    name_isf  (Nisf) <U14 504B 'Ross' 'Filchner-Ronne' ... 'Moscow Univ.'
    Tp        (Nisf, regime) float64 216B -1.837 -1.974 -1.684 ... 0.0 0.0
    Sp        (Nisf, regime) float64 216B 33.22 36.7 34.5 ... 33.97 0.0 0.0
    Td        (Nisf, regime) float64 216B 1.772 -1.955 -1.597 ... 1.164 0.0 0.0
    Sd        (Nisf, regime) float64 216B 34.69 36.69 34.5 ... 34.66 0.0 0.0
    Tc        (Nisf, regime, box) float64 1kB -1.055 -1.752 -1.976 ... 0.0 0.0
    Sc        (Nisf, regime, box) float64 1kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    melt      (Nisf, regime) float64 216B 0.2256 0.001004 0.004368 ... 0.0 0.0
    overt     (Nisf, regime) float64 216B 0.8345 0.1263 0.1767 ... 0.0 0.0
    qAABW     (Nisf) float64 72B 0.4413 0.05799 0.1463 ... 0.08865 0.09817 0.0
Attributes:
    Global:   Dataset containing the output parameters for the box model
    Nisf:     Index of the ice shelves (from Burgard 2022)
    box:      box number (but not ntotal box number)

### Generate Tab2 (SI)

In [18]:
index2name = {10: "Ross",
              11: "Filchner-Ronne",
              31: "Amery",
              52: "Dotson",
              66: "Pine Island",
              43: "Riiser-Larsen",
              48: "Totten",
              45: "Baudouin",
              33: "Moscow Univ."
             }

for index in Dataset.Nisf.values:
    data_isf = Dataset.sel(Nisf=index)
    print(index2name[index]+' & '+
          str(np.round(data_isf.melt.sel(regime='diff').values*12,2))+' & '+ #*12 for yearly
          str(np.round(data_isf.overt.sel(regime='diff').values,2))+' & '+
          str(np.round(data_isf.melt.sel(regime='conv').values*12,2))+' & '+ #*12 for yearly
          str(np.round(data_isf.overt.sel(regime='conv').values,2))+' & '+
          str(np.round(data_isf.qAABW.values,2))+' &'
         )

Ross & 2.71 & 0.83 & 0.01 & 0.13 & 0.44 &
Filchner-Ronne & 1.17 & 0.6 & 0.06 & 0.23 & 0.06 &
Amery & 8.06 & 0.51 & 0.58 & 0.19 & 0.15 &
Dotson & 0.0 & 0.0 & 1.11 & 0.05 & 0.14 &
Pine Island & 24.74 & 0.23 & 3.79 & 0.1 & 0.03 &
Riiser-Larsen & 6.66 & 0.3 & 0.67 & 0.12 & 0.01 &
Baudouin & 7.07 & 0.26 & 0.26 & 0.07 & 0.09 &
Totten & 20.58 & 0.25 & 3.22 & 0.11 & 0.1 &
Moscow Univ. & 18.78 & 0.17 & 0.0 & 0.0 & 0.0 &


In [19]:
for index in Dataset.Nisf.values:
    data_isf = Dataset.sel(Nisf=index)
    Ac = inputs.sel(Nisf=index).Ac.values
    print(index2name[index]+' & '+
          str(np.round(data_isf.melt.sel(regime='diff').values*12 *Ac*rho_i/1e12,1))+' & ' +#*12 for yearly
          str(np.round(data_isf.melt.sel(regime='conv').values*12 *Ac*rho_i/1e12,1))+' & ' #*12 for yearly
         )
    

Ross & 1155.8 & 5.1 & 
Filchner-Ronne & 444.3 & 23.1 & 
Amery & 431.8 & 30.8 & 
Dotson & 0.0 & 5.0 & 
Pine Island & 129.0 & 19.7 & 
Riiser-Larsen & 253.5 & 25.3 & 
Baudouin & 212.1 & 7.7 & 
Totten & 123.2 & 19.3 & 
Moscow Univ. & 96.9 & 0.0 & 
